# Rung 18 — one training run, two data levers, three orthogonal readouts**Baseline.** rung 06 **ep3** (`merged/checkpoint-2580`, `bucket_mean` **0.5724**) — theepoch-matched control that closed rungs 14 and 15. The base model is not a comparator.**The two levers are both DATA; the recipe does not move.**| lever | what it changes | read by ||---|---|---|| **L1** minted zero-count rows, `number` only | adds ~670 rows whose gold is `0` | probe **16a** `zero_probe`, on a held-out slice || **L2** stem paraphrase + `"Please provide a number."` dropout | rewrites wording, adds no labels | probe **16d** `format_audit` || both | — | canonical eval + the **new** count-rank metric |⚠️ One run cannot recover a factorial design: an L1xL2 interaction is unattributable. Itdoes mean a null is diagnosable rather than mute, because the two readouts do not touch.**Pre-registered win condition.** Spearman **r** of the predicted count vs gold on the`Clips` template must RISE **and** `margin_OOD` must not fall. The model sits at **0.43**where a blind human scores **0.7230** under the same gold([[gold-is-signal-model-underuses-it]]) — the scale is already close, so mean error hidesthis and only a rank metric can see it.**Utilisation, not recipe.** `per_device 1 x grad_accum 16` becomes `4 x 4`: effective batch**16**, identical. `assert_recipe_unchanged` proves it against rung 06's own argv.> This notebook stops after training. Per-epoch merge + eval + scoring live in> `18b_epoch_eval.ipynb` — every epoch is evaluated (RULES §6b): rung 06 erased counting> monotonically across epochs and a single end-of-run eval would have hidden it.

In [ ]:
# --- bootstrap -----------------------------------------------------------------
import json, logging, os, sys, time
from dataclasses import replace
from pathlib import Path

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent

# `swift` is shelled out to by _train / merge. A papermill kernel does NOT inherit the
# env's bin/ on PATH, and it must be THIS interpreter's bin so the CLI and the kernel come
# from one environment.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")

for p in (REPO / "src", REPO / "vendor" / "orena-focus" / "src",
          EXP / "_models", EXP / "_tools",
          REPO / "experiments" / "16-count-probes" / "_models",   # zero_probe  (probe 16a)
          REPO / "experiments" / "02-lora-sft" / "_models"):      # lora_sft_train (rung 02)
    if p.is_dir():
        sys.path.insert(0, str(p))

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

import torch
print("torch", torch.__version__, "|",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "->", _envbin)
print("repo:", REPO)


In [ ]:
# --- parameters (RAW LITERALS ONLY — papermill injects its overrides BELOW this cell) ----
SMOKE = True          # full run: -p SMOKE False
SEED  = 42

RUN       = "18_count_aug_v1"
SMOKE_RUN = "18_count_aug_smoke"

# L1 — minted zero-count rows, `number` only (16a: `binary` already carries the concept)
MINT_ZEROS    = True
DOSE_RATIO    = 2.5    # real per-class count rows : minted zeros (PLAN band 3:1-2:1)
UNSEEN_SHARE  = 0.10   # capped share for classes with zero examples anywhere
MAX_PER_FRAME = 2

# L2 — stem paraphrase + format-tail dropout, `number` only
PARAPHRASE   = True
STEM_RATE    = 0.40
TAIL_DROPOUT = 0.25

# utilisation (effective batch stays 16 = rung 06's)
# 🔴 MEASURED on the 32 GB RTX 5090, 2026-07-28 (runs/18_count_aug_smoke/vram_probe.csv):
#   pd=2 -> 26,813 MiB peak, OK, 13.66 s/it   |   pd=4 -> OOM at 32,076   |   pd=6 -> OOM
# So the PLAN's `4 x 4` does not fit this card and `2 x 8` is the largest that does. The
# effective batch is 16 either way, which is the only thing comparability depends on.
PER_DEVICE  = 2
GRAD_ACCUM  = 8
# 1 is rung 06's own micro-batch: probing it is what turns "2 is what fits" into a MEASURED
# utilisation gain rather than an assumed one. 4 and 6 are not re-probed — they are dead.
VRAM_PROBE  = [1, 2]
PROBE_STEPS = 6        # >2: rung 06's first smoke read 110 s/it and that was all warm-up
SMOKE_STEPS = 4

DATA_ROOT    = "/workspace/orena-data"
MODEL_BASE   = "/workspace/models/qwen3-vl-8b"
FRAMES_CACHE = "/workspace/frames_cache"
RUNG06_JSONL = "/workspace/repo/experiments/06-vit-lora/runs/06_vit_lora_v1/train.jsonl"


In [ ]:
# --- derived (MUST live BELOW the parameters cell) ------------------------------
# 🔴 papermill injects its override cell immediately AFTER the cell tagged `parameters`.
# A value DERIVED inside that cell is computed from the pre-injection literals and is never
# recomputed, so `-p SMOKE False` changes SMOKE and nothing else. Rung 16's first "full"
# run was the smoke again (n=64) and was indistinguishable from a success. Every derived
# value lives here, below the injection point, and gate G0 asserts the REALIZED artifact
# matches the declared mode.
RUN_NAME = SMOKE_RUN if SMOKE else RUN
RUN_DIR  = EXP / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST = REPO / "experiments" / "splits" / "frame_ood_v1.csv"

print(f"SMOKE      {SMOKE}   (steps={SMOKE_STEPS})")
print(f"run dir    {RUN_DIR}")
print(f"L1 mint    {MINT_ZEROS}  dose 1:{DOSE_RATIO}  unseen {UNSEEN_SHARE}  cap/frame {MAX_PER_FRAME}")
print(f"L2 phrase  {PARAPHRASE}  stem {STEM_RATE}  tail-dropout {TAIL_DROPOUT}")
print(f"batch      {PER_DEVICE} x {GRAD_ACCUM} = {PER_DEVICE * GRAD_ACCUM}")


## Gate orderEverything gold-only runs first, and nothing loads a model until it has passed. Amisconfigured run must abort in seconds, not after a 17 GB load or an export.1. **recipe** — rung 18 differs from rung 06 only in the batch re-pack, product unchanged2. **byte-identity** — with both levers OFF the export equals rung 06's `train.jsonl`3. **dose / class-mix / val-leak / held-out-slice** — inside the build (they need the items)4. **L2 rates** — the realized paraphrase + dropout rates equal the requested ones5. **G0 mode** — the realized row count matches SMOKE6. **G1** — the LoRA actually reaches the ViT (read from swift's own log)

In [ ]:
# --- GATE 1: the recipe did not move -------------------------------------------
from count_aug_train import (CountAugConfig, assert_recipe_unchanged, effective_batch,
                             measure_vram, project_hours, read_g1, list_checkpoints, _train)

cfg = CountAugConfig(
    exp_dir=EXP, run_name=RUN_NAME, model_path=Path(MODEL_BASE), data_root=Path(DATA_ROOT),
    manifest_path=MANIFEST, per_device_train_batch_size=PER_DEVICE,
    gradient_accumulation_steps=GRAD_ACCUM, seed=SEED,
    smoke=SMOKE, smoke_max_steps=SMOKE_STEPS,
)
cfg.val_jsonl = cfg.run_dir / "val.jsonl"      # eval_loss only, exactly as rung 06

g = assert_recipe_unchanged(cfg)
for flag, (r06, r18) in sorted(g["diff"].items()):
    print(f"  {flag:36s} rung06={r06!s:6s} -> rung18={r18}")
print(f"\nOK GATE 1: effective batch {g['effective_batch']} (== rung 06), recipe otherwise identical")


In [ ]:
# --- load the corpus ONCE (the parquet parse is the slow part) ------------------
import build_train_jsonl as btj

BUILD = btj.BuildConfig(
    data_root=Path(DATA_ROOT), model_path=Path(MODEL_BASE), manifest_path=MANIFEST,
    frames_dir=Path(FRAMES_CACHE), rung06_jsonl=Path(RUNG06_JSONL), seed=SEED,
)
t0 = time.perf_counter()
ITEMS = btj.load_items(BUILD)
print(f"{len(ITEMS)} FRAME items in {time.perf_counter() - t0:.0f}s")


In [ ]:
# --- GATE 2: flags OFF must be BYTE-IDENTICAL to rung 06's train.jsonl -----------
# 🔴 The load-bearing gate. This module re-implements rung 02's serialization so the levers
# can be inserted into it; without proving the re-implementation, a one-character drift is
# indistinguishable from a lever. Built at FULL size even in SMOKE (gold-only, no GPU) —
# a byte-identity gate on 64 rows proves nothing.
ctrl = replace(BUILD, mint_zeros=False, paraphrase=False, smoke=False,
               out_jsonl=RUN_DIR / "train_flagoff.jsonl")
t0 = time.perf_counter()
ctrl_stats = btj.build(ctrl, items=ITEMS)
print(f"control build: {ctrl_stats['n_export_rows']} rows in {time.perf_counter() - t0:.0f}s")

btj.assert_flag_off_identical(ctrl.out_jsonl, Path(RUNG06_JSONL))

# Temp artifact: deleted the moment it is not needed (CONSTITUTION §IX). Its sha256 is
# recorded in build_stats.json, so the gate stays reproducible without keeping 40 MB around.
ctrl.out_jsonl.unlink()
print("OK GATE 2: flags-off == rung 06, control file removed")


In [ ]:
# --- build train.jsonl with BOTH levers (gates 3 + 4 fire inside) ---------------
BUILD_ON = replace(
    BUILD, mint_zeros=MINT_ZEROS, paraphrase=PARAPHRASE, dose_ratio=DOSE_RATIO,
    unseen_share=UNSEEN_SHARE, max_per_frame=MAX_PER_FRAME, stem_rate=STEM_RATE,
    tail_dropout=TAIL_DROPOUT, smoke=SMOKE, out_jsonl=cfg.train_jsonl,
)
stats = btj.build(BUILD_ON, items=ITEMS)
stats["control_sha256"] = ctrl_stats["sha256"]
(RUN_DIR / "build_stats.json").write_text(json.dumps(stats, indent=2, default=str))

print(f"rows      {stats['n_export_rows']}  ({stats['n_real_rows']} real + {stats['n_minted_rows']} minted)")
if MINT_ZEROS:
    m = stats["mint"]
    print(f"dose      {m['realized']}/{m['target']} against {m['n_perclass_rows']} real per-class rows")
    print(f"mix       {m['class_mix']}")
    print(f"evidence  {m['n_confirmed_by_binary']} zeros positively confirmed by the co-occurrence binaries")
    print(f"repaired  {m['n_refuted_by_binary']} would-be-WRONG zeros refuted (the 8.4% under-naming, caught)")
    print(f"unseen    {m['n_unseen_minted']} rows on classes with zero examples anywhere")
if PARAPHRASE:
    print(f"L2        {stats['paraphrase_audit']}")


In [ ]:
# --- GATE 0: the realized artifact must match the DECLARED mode -----------------
# The rung-16 trap made concrete. A parameter that fails to take effect is otherwise
# indistinguishable from a successful run, so the check is on the ARTIFACT, never on the
# variable: `SMOKE` could be False while the file is still the smoke's.
_small = stats["n_export_rows"] < 1000
assert _small == SMOKE, (
    f"MODE GATE FAILED: SMOKE={SMOKE} but the export has {stats['n_export_rows']} rows. "
    "Either -p SMOKE False did not take effect (papermill injects BELOW the parameters "
    "cell — check that no derived value is computed inside it), or the build is wrong."
)
assert stats["mint_zeros"] == MINT_ZEROS and stats["paraphrase"] == PARAPHRASE
print(f"OK GATE 0: {stats['n_export_rows']} rows is consistent with SMOKE={SMOKE}")

import pandas as pd
if MINT_ZEROS:
    mdf = pd.read_csv(cfg.train_jsonl.with_name("minted.csv"))
    print("\n--- minted rows, eyeball (user rule: error examples every run) ---")
    print(mdf.groupby(["cls", "evidence"]).size().unstack(fill_value=0).to_string())
    print()
    print(mdf[["dataset", "cls", "question", "answer", "named", "evidence"]].head(8).to_string(index=False))


In [ ]:
# --- val.jsonl for eval_loss (observability only, exactly as rung 06) -----------
# Held out from OUR val split, never carved from train: verified on ms-swift 4.4.1 that
# split_dataset_ratio defaults to 0.0 and is bypassed when --val_dataset is passed.
from frame import split as sp
from frame.config import BaselineConfig
from frame.data import FrameProvider, frame_cache_name
from frame.engine import SYSTEM_PROMPT

bcfg = BaselineConfig(data_root=cfg.data_root, model_path=cfg.model_path,
                      datasets=cfg.datasets, base_fps=cfg.base_fps,
                      max_pixels=cfg.max_pixels, seed=cfg.seed)
val_items = sp.apply_split(ITEMS, sp.load_manifest(cfg.manifest_path), "val_ood")
val_items = sorted(val_items, key=lambda it: (it.dataset, it.video_id, it.frame_index))[:256]

provider = FrameProvider(bcfg)
with open(cfg.val_jsonl, "w", encoding="utf-8") as fh:
    for it in val_items:
        img = cfg.frames_dir / frame_cache_name(it)
        if not img.exists():
            provider.ensure_reader(it)
            provider.get_frame(it).save(img, quality=95)
        fh.write(json.dumps({"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"<image>{it.request.question}"},
            {"role": "assistant", "content": str(it.reference.answer)},
        ], "images": [str(img)]}, ensure_ascii=False) + "\n")
provider.close()
print(f"val.jsonl -> {cfg.val_jsonl} ({len(val_items)} examples)")


In [ ]:
# --- MEASURE the batch size before committing the card --------------------------
# 32 GB RTX 5090, 8B bf16 + LoRA + activations. `per_device=4` is a fit-or-OOM question and
# an OOM three hours in costs the whole run, so it is measured on 3 real steps rather than
# assumed. grad_accum moves with it so the effective batch stays 16 in every probe: the
# probe measures the configuration we would actually run.
#
# An OOM here is a RESULT, not a crash — that is the point of the probe.
probe_rows = []
for pd_size in VRAM_PROBE:
    probe_rows.append(measure_vram(cfg, pd_size, steps=PROBE_STEPS))

probe_df = pd.DataFrame(probe_rows)
# Projected against the FULL row count, never the smoke's: `n_real_rows` is the whole train
# split by construction (SMOKE subsets only the export, after the levers and their gates).
_full_rows = stats["n_real_rows"] + stats["n_minted_rows"]
probe_df["proj_h"] = [project_hours(r["s_per_sample"], _full_rows, cfg) for r in probe_rows]
_total_mib = torch.cuda.get_device_properties(0).total_memory // (1024 ** 2)
probe_df["headroom_mib"] = _total_mib - probe_df["peak_mib"]
probe_df.to_csv(RUN_DIR / "vram_probe.csv", index=False)
print(probe_df.to_string(index=False))

if SMOKE:
    print("\n⚠️  SMOKE: the probe trained on the 64-row smoke export, so its peak is a WIRING")
    print("    check, not a memory verdict — the longest sequences may not be in that sample.")
    print("    The full pass re-runs this against the real train.jsonl.")

chosen = probe_df[probe_df.per_device == PER_DEVICE]
assert len(chosen) and bool(chosen.ok.iloc[0]), (
    f"VRAM GATE FAILED: per_device={PER_DEVICE} did not survive {PROBE_STEPS} steps "
    f"({chosen.error.iloc[0] if len(chosen) else 'not probed'}). Do NOT lower max_pixels or "
    "the effective batch to make it fit — drop per_device to the largest size that passed "
    "and raise grad_accum to keep the product at 16."
)
print(f"\nOK: per_device={PER_DEVICE} fits with {int(chosen.headroom_mib.iloc[0])} MiB spare")


In [ ]:
# --- train ----------------------------------------------------------------------
# STOP RULES (unchanged from rung 06): OOM -> STOP; do NOT lower max_pixels / LR / effective
# batch, each is a second variable. Loss diverges -> report it, do not touch the LR.
# The broken-run guard (rung 06's) is ON for the full run and OFF in SMOKE: it is a pure
# stdout observer, so when it does not abort the run is byte-identical to an unguarded one.
t0 = time.perf_counter()
_train(cfg)
print(f"\ntraining done in {(time.perf_counter() - t0) / 3600:.2f} h")
print("checkpoints:", [c.name for c in list_checkpoints(cfg)])


In [ ]:
# --- GATE 5 (G1): did the LoRA actually reach the ViT? --------------------------
# Read from swift's OWN log, which prints both numbers before the trainer starts. It matters
# because tuner.py:93 has an early return that ignores freeze_vit in silence; it does not
# fire on 4.4.1 (the CLI parses --target_modules to a list) but this is the runtime proof.
g1 = read_g1(cfg)
print(f"  model_parameter_info : {g1['model_parameter_info']}")
print(f"  trainable (M)        : {g1['trainable_params_M']}")
print(f"  targets vision_tower : {g1['targets_vision_tower']}")

assert g1["trainable_params_M"] is not None, "G1 FAILED: swift never logged model_parameter_info"
assert g1["targets_vision_tower"], (
    "G1 FAILED: the LoRA targets contain NO vision_tower module — this run is rung 02's "
    "model under a new name, and its comparison to rung 06 is void.")
assert g1["trainable_params_M"] < 500, (
    f"G1 FAILED: {g1['trainable_params_M']}M trainable — a FINE-TUNE, not LoRA.")
print("\nOK G1: LoRA (not fine-tune), and it reaches the vision tower")


## 🚦 Stops hereTraining is done and every gate has passed. Scoring is `18b_epoch_eval.ipynb`, run once perepoch — **an unevaluated epoch is a missing control, not a discarded one** (RULES §6b, andthe reason rungs 14 and 15 both compared against the wrong rung-06 epoch).Read, in this order:1. **Spearman r** on the `Clips` template vs rung 06 ep3's **0.43** (human: **0.7230**)2. **`margin_OOD`** — pre-registered as a no-fall condition, not a nice-to-have3. `bucket_mean` vs **0.5724**4. probe **16a** zero-emission on the held-out slice → did **L1** land?5. probe **16d** illegal rate on `number`, with `binary`/`fo_class` as the untouched   within-run control → did **L2** land?